In [2]:
from pypdf import PdfReader
import unicodedata
import re
import os
from pathlib import Path

Os pdf's feitos em latex dos professores continham acentos com códigos diferentes do convencional (não eram os mesmos do teclado comum), então fiz a substituição pro algoritimo não ler o texto de maneira errada (Ex: 'cálculo' virava 'c ´alculo)

In [3]:
# acento do pdf -> acento combinante padrão
acentuacao = {
    '\u00b4': '\u0301',  # acento agudo
    '\u0060': '\u0300',  # acento grave
    '\u02dc': '\u0303',  # til
    '\u02c6': '\u0302',  # acento circunflexo
    '\u00b8': '\u0327',  # ç
}

def corrigir_texto(texto):
    for acento, combinante in acentuacao.items():
        texto = re.sub(re.escape(acento) + r'\s*(.)', r'\1' + combinante, texto)
    return unicodedata.normalize('NFC', texto) # fundindo o texto com o acento

In [8]:
# tentei criar as chuncks manualmente pro modelo de embeddings mas estava dando muitos erros, dai descobri essa
# biblioteca que corta o texto de maneira natural
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
pasta_pdfs = Path("material")
chunks_totais = []

# lendo e corrigindo

for arquivo_pdf in pasta_pdfs.glob("*.pdf"):
    reader = PdfReader(str(arquivo_pdf))

    texto_completo = ''
    for pagina in reader.pages:
        texto_completo += pagina.extract_text()

    texto_corrigido = corrigir_texto(texto_completo)
    chunks_arquivo = splitter.split_text(texto_corrigido)

In [ ]:
# usei auxilio de IA para encontrar um bom modelo, esse foi escolhido por lidar bem com multiplos idiomas
# incluindo o portugues

from sentence_transformers import SentenceTransformer

modelo_embedding = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda') # ativei a gpu, do contrario iria demorar demais.

# o modelo exige prefixar o texto com "passage: " para documentos
# e "query: " para perguntas.
textos_prefixados = [f"passage: {chunk}" for chunk in chunks_totais]

embeddings = modelo_embedding.encode(textos_prefixados, show_progress_bar=True)

In [ ]:
# chromadb = biblioteca pra salvar os vetores no disco
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

colecao = client.get_or_create_collection(name="materiais_estudo")

ids = [f"chunk_{i}" for i in range(len(chunks_totais))]

colecao.add(
    ids=ids, embeddings=embeddings.tolist(),
    documents=chunks_totais
)

In [ ]:
# testando os chuncks
pergunta = "O que é cálculo?"
embedding_pergunta = modelo_embedding.encode(f"query: {pergunta}")

resultados = colecao.query(
    query_embeddings=[embedding_pergunta.tolist()],
    n_results=3) # pedi 3 chuncks parecidas

print(resultados['documents'][0][0])
print(resultados['documents'][0][1])
print(resultados['documents'][0][2])

In [ ]:
# instaei o ollama para gerar a resposta
import ollama

def responder_pergunta(pergunta, n_chunks=3):
    # pegando os chuncks mais parecidos
    embedding_pergunta = modelo_embedding.encode(f"query: {pergunta}")
    resultados = colecao.query(
        query_embeddings=[embedding_pergunta.tolist()],
        n_results=n_chunks
    )

    chunks_relevantes = resultados['documents'][0]
    contexto = "\n\n---\n\n".join(chunks_relevantes)
    comando = f"""Responda a pergunta do usuário usando APENAS as informações do contexto abaixo.
Se a resposta não estiver no contexto, diga que não encontrou essa informação nos materiais.
{contexto}
{pergunta}
"""
# enviando
    resposta = ollama.generate(model='llama3.1:8b', prompt=comando)
    return resposta['response']

# teste final
resposta = responder_pergunta("O que é cálculo?")
print(resposta)

In [ ]:
resposta = responder_pergunta("O que é regressão linear?")
print(resposta)